# 🤖 DistilBERT Fine-Tuning — Occupation Classifier
**Project:** Mitigating Algorithmic Bias in AI-Powered Resume Screening

Pipeline:
1. Load pre-split tokenized datasets (train/val/test)
2. Label encoding
3. DistilBERT fine-tuning
4. Evaluation (accuracy, F1, confusion matrix)
5. Save model + suitability scores


## 0. Imports & Config

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_CSV  = Path('../train_tokenized.csv')   # balanced, from preprocessing
VAL_CSV    = Path('../val_tokenized.csv')
TEST_CSV   = Path('../test_tokenized.csv')
MODEL_DIR  = Path('../model')
SCORES_CSV = Path('../suitability_scores.csv')
MODEL_DIR.mkdir(exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LEN    = 512
BATCH_SIZE = 16
EPOCHS     = 4
LR         = 2e-5
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config ready ✓')


Device : cpu
PyTorch: 2.12.0+cpu
Config ready ✓


## 1. Load Tokenized Dataset

In [9]:
# Load pre-split tokenized datasets — no leakage (split was done at resume level)
def load_tokenized(path):
    df = pd.read_csv(path)
    df['input_ids']      = df['input_ids'].apply(json.loads)
    df['attention_mask'] = df['attention_mask'].apply(json.loads)
    return df

train_df = load_tokenized(TRAIN_CSV)
val_df   = load_tokenized(VAL_CSV)
test_df  = load_tokenized(TEST_CSV)

print(f'Train : {len(train_df):>6,} chunks ({train_df["occupation"].nunique()} classes)')
print(f'Val   : {len(val_df):>6,} chunks')
print(f'Test  : {len(test_df):>6,} chunks')
print(f'\nChunks per occupation (train):')
print(train_df['occupation'].value_counts().to_string())


Train :  6,614 chunks (24 classes)
Val   :  1,322 chunks
Test  :  1,258 chunks

Chunks per occupation (train):
occupation
INFORMATION-TECHNOLOGY    325
CONSULTANT                318
HEALTHCARE                317
BPO                       307
AGRICULTURE               304
ADVOCATE                  297
PUBLIC-RELATIONS          294
ENGINEERING               290
HR                        287
DIGITAL-MEDIA             285
BANKING                   282
APPAREL                   278
ACCOUNTANT                276
AVIATION                  274
AUTOMOBILE                268
CHEF                      266
FINANCE                   266
BUSINESS-DEVELOPMENT      258
CONSTRUCTION              255
ARTS                      242
FITNESS                   238
SALES                     237
TEACHER                   228
DESIGNER                  222


## 2. Label Encoding

In [10]:
# Fit LabelEncoder on train only, then apply to all splits
le = LabelEncoder()
le.fit(train_df['occupation'])

train_df['label'] = le.transform(train_df['occupation'])
val_df['label']   = le.transform(val_df['occupation'])
test_df['label']  = le.transform(test_df['occupation'])

NUM_CLASSES = len(le.classes_)
print(f'Number of classes: {NUM_CLASSES}')
print(f'\nLabel mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i:>2}  {cls}')

# Save label mapping
pd.DataFrame({'label': range(NUM_CLASSES), 'occupation': le.classes_}).to_csv(
    Path('../label_map.csv'), index=False
)
print('\nLabel map saved → ../label_map.csv')


Number of classes: 24

Label mapping:
   0  ACCOUNTANT
   1  ADVOCATE
   2  AGRICULTURE
   3  APPAREL
   4  ARTS
   5  AUTOMOBILE
   6  AVIATION
   7  BANKING
   8  BPO
   9  BUSINESS-DEVELOPMENT
  10  CHEF
  11  CONSTRUCTION
  12  CONSULTANT
  13  DESIGNER
  14  DIGITAL-MEDIA
  15  ENGINEERING
  16  FINANCE
  17  FITNESS
  18  HEALTHCARE
  19  HR
  20  INFORMATION-TECHNOLOGY
  21  PUBLIC-RELATIONS
  22  SALES
  23  TEACHER

Label map saved → ../label_map.csv


In [11]:
# Splits were done in Preprocessing.ipynb at resume level before tokenization
# Verify no filename leakage across splits
train_files = set(train_df['filename'])
val_files   = set(val_df['filename'])
test_files  = set(test_df['filename'])

assert len(train_files & test_files) == 0, 'Leakage: train/test overlap!'
assert len(val_files   & test_files) == 0, 'Leakage: val/test overlap!'
print('Leakage check passed ✓')
print(f'  Train resumes : {train_df["filename"].nunique()}')
print(f'  Val resumes   : {val_df["filename"].nunique()}')
print(f'  Test resumes  : {test_df["filename"].nunique()}')


Leakage check passed ✓
  Train resumes : 1495
  Val resumes   : 372
  Test resumes  : 373


## 4. PyTorch Dataset & DataLoaders

In [12]:
class ResumeDataset(Dataset):
    def __init__(self, dataframe):
        self.input_ids      = torch.tensor(dataframe["input_ids"].tolist(),      dtype=torch.long)
        self.attention_mask = torch.tensor(dataframe["attention_mask"].tolist(), dtype=torch.long)
        self.labels         = torch.tensor(dataframe["label"].tolist(),          dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

train_dataset = ResumeDataset(train_df)
val_dataset   = ResumeDataset(val_df)
test_dataset  = ResumeDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")


Train batches : 414
Val batches   : 83
Test batches  : 79


## 5. Load DistilBERT Model

In [13]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES,
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7692.58it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params    : 66,971,928
Trainable params: 66,971,928


## 6. Optimizer & Scheduler

In [14]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(0.1 * total_steps)   # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total training steps : {total_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"Learning rate        : {LR}")


Total training steps : 1656
Warmup steps         : 165
Learning rate        : 2e-05


## 7. Fine-Tuning

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs     = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds       = outputs.logits.argmax(dim=-1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total


# ── Training loop ─────────────────────────────────────────────────────────────
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss,   val_acc   = eval_epoch(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(MODEL_DIR)
        print(f"  ✓ Best model saved (val_acc={best_val_acc:.4f})")


## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history["train_loss"], label="Train", marker="o")
axes[0].plot(epochs_range, history["val_loss"],   label="Val",   marker="o")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train", marker="o")
axes[1].plot(epochs_range, history["val_acc"],   label="Val",   marker="o")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.suptitle("DistilBERT Fine-Tuning", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 9. Evaluation on Test Set

In [ ]:
# Load best model
best_model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
best_model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=-1)
        preds   = probs.argmax(dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print(f"Test Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Macro F1      : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"Weighted F1   : {f1_score(all_labels, all_preds, average='weighted'):.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=le.classes_))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title("Confusion Matrix — Test Set", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()


## 10. Suitability Scores

In [ ]:
# Suitability Score = softmax probability of the predicted class
# Represents model confidence in the occupation classification

suitability_scores = all_probs.max(axis=1)   # highest prob across all classes
predicted_labels   = le.inverse_transform(all_preds)
true_labels        = le.inverse_transform(all_labels)

scores_df = test_df[["filename", "occupation", "chunk_id"]].copy().reset_index(drop=True)
scores_df["predicted_occupation"] = predicted_labels
scores_df["suitability_score"]    = suitability_scores.round(4)
scores_df["correct"]              = (all_preds == all_labels)

scores_df.to_csv(SCORES_CSV, index=False)

print(f"Saved: {SCORES_CSV}")
print(f"\nSuitability Score stats:")
print(scores_df["suitability_score"].describe().round(4))
print(f"\nSample:")
print(scores_df[["filename", "occupation", "predicted_occupation",
                  "suitability_score", "correct"]].head(10).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
axes[0].hist(scores_df["suitability_score"], bins=30,
             color="#4C9BE8", edgecolor="white")
axes[0].set_title("Suitability Score Distribution", fontweight="bold")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")

# Correct vs incorrect predictions
scores_df.groupby("correct")["suitability_score"].plot(
    kind="kde", ax=axes[1], legend=True
)
axes[1].set_title("Score Distribution: Correct vs Incorrect", fontweight="bold")
axes[1].set_xlabel("Suitability Score")
axes[1].legend(["Incorrect", "Correct"])

plt.tight_layout()
plt.show()
